In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import uuid

spark = SparkSession.builder.getOrCreate()


Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1764899837815_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
S3_ECOMMERCE_PATH = "s3://msba-bigdata-groupproject/data/ecommerce/*.csv"
S3_CUSTOMERS_PATH = "s3://msba-bigdata-groupproject/data/customers.csv"
S3_OUTPUT_PATH    = "s3://msba-bigdata-groupproject/dynamodb/session_kv_clean/"
TARGET_ROWS       = 1500000


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
df = (
    spark.read
    .option("header", True)
    .csv(S3_ECOMMERCE_PATH)
    .withColumn("event_time", F.to_timestamp("event_time"))
)

df.show(5)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code| brand| price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+------+------+---------+--------------------+
|2019-11-01 00:00:00|      view|   1003461|2053013555631882655|electronics.smart...|xiaomi|489.07|520088904|4d3b30da-a5e4-49d...|
|2019-11-01 00:00:00|      view|   5000088|2053013566100866035|appliances.sewing...|janome|293.65|530496790|8e5f4f83-366c-4f7...|
|2019-11-01 00:00:01|      view|  17302664|2053013553853497655|                NULL| creed| 28.31|561587266|755422e7-9040-477...|
|2019-11-01 00:00:01|      view|   3601530|2053013563810775923|appliances.kitche...|    lg|712.87|518085591|3bfb58cd-7892-48c...|
|2019-11-01 00:00:01|      view|   1004775|2053013555631882655|electronics.smart...|xiaomi

In [4]:
customers = (
    spark.read
    .option("header", True)
    .csv(S3_CUSTOMERS_PATH)
    .select("customer_id")
)

customer_ids = [r.customer_id for r in customers.collect()]
broadcast_customers = spark.sparkContext.broadcast(customer_ids)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
@F.udf("string")
def random_customer():
    import random
    return random.choice(broadcast_customers.value)

df = df.withColumn("customer_id", random_customer())


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
df = (
    df.select(
        F.col("user_session").alias("session_id"),
        "customer_id",
        "product_id",
        "event_type",
        "event_time",
        "price",
        "brand",
        "category_code",
        "category_id"
    )
)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
def sanitize(col):
    return (
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(
                        F.regexp_replace(F.col(col), r'[\r\n]+', ' '),
                        '"', "'"
                    ),
                    ',', ';'
                ),
                r'\t', ' '
            ),
            r'[\x00-\x1F\x7F]', ''
        )
    )

df = (
    df.withColumn("brand", sanitize("brand"))
      .withColumn("category_code", sanitize("category_code"))
)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [8]:
df = df.filter(F.col("session_id").isNotNull())


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
df = df.limit(TARGET_ROWS)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
df_final = df.select(
    "session_id",
    "customer_id",
    "product_id",
    "event_type",
    "event_time",
    "price",
    "brand",
    "category_code",
    "category_id"
)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
(
    df_final.coalesce(1)
    .write
    .option("header", True)
    .option("quote", "")
    .option("escape", "")
    .mode("overwrite")
    .csv(S3_OUTPUT_PATH)
)


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…